# Notebook 06 — Model Final & Export untuk Produksi

**Skripsi: Sistem Prediksi Risiko Diabetes (DiaPredict)** — Revisi Pengujian V3

---

## Peran notebook ini

Notebook ini adalah **penutup rantai revisi**. Notebook 01–05 menjawab pertanyaan
penguji satu per satu (kenapa rasio split 80:20, kenapa `k` KNN sekian, kenapa
hyperplane SVM seperti itu, apakah selisih antar model signifikan secara
statistik, dan seberapa tahan model terhadap perubahan asumsi). Notebook 06
**tidak menambah eksperimen baru** — tugasnya adalah:

1. **Mengumpulkan** seluruh hasil notebook 01–05 dari folder `json/`
   (kontrak nama file ada di `_SPEC_BERSAMA.md` bagian 2).
2. **Melatih model final** memakai konfigurasi yang sudah terjustifikasi —
   bukan angka default, melainkan nilai yang dipilih karena ada buktinya.
3. **Mengekspor artefak produksi** (`rf_model.pkl`, `scaler.pkl`,
   `model_metadata.json`) yang dibaca langsung oleh website Laravel DiaPredict.
4. **Menghasilkan `experiments.json`**, satu file yang dibaca halaman
   "Metodologi & Pengujian" di website supaya angka di web = angka di skripsi.

## Tiga masalah produksi yang diperbaiki notebook ini

| Masalah pada sistem lama | Perbaikan di notebook 06 |
|---|---|
| **Urutan fitur tertukar.** Kode inferensi lama menyusun input sebagai `[age, hypertension, bmi, HbA1c, glukosa]`, padahal model dilatih dengan urutan `[age, bmi, hypertension, HbA1c, glukosa]`. BMI dan Hipertensi saling tertukar setiap kali prediksi. | Mengekspor `model_metadata.json` berisi field **`feature_order`** eksplisit, plus **uji regresi** (CELL 10) yang membuktikan besarnya dampak bug tersebut. |
| **Model produksi bukan model hasil tuning.** `train_model.py` memakai `RandomForestClassifier(n_estimators=100)` default tanpa `max_depth`, sehingga pohon tumbuh sampai daun murni dan pickle-nya membengkak ~78 MB. | Model final memakai hyperparameter hasil tuning (`max_depth=10`, `min_samples_leaf=4`, dst.) sehingga ukuran file turun drastis dan sesuai dengan yang dilaporkan di skripsi. |
| **Batas winsorization tidak diketahui server.** Preprocessing di notebook melakukan capping IQR, sedangkan inferensi produksi mengirim nilai mentah. | Batas capping per fitur diekspor ke `model_metadata.json` supaya `predict.py` bisa menerapkan capping yang sama. |

> **Catatan urutan menjalankan:** notebook ini membaca hasil notebook 01–05.
> Jalankan notebook `00`–`05` terlebih dahulu dengan `PAKAI_DRIVE = True`
> agar file JSON-nya tersimpan permanen dan terbaca di sini. Kalau ada file
> yang belum tersedia, notebook ini **tetap berjalan** memakai nilai cadangan
> (baseline V2) dan menandai bagian tersebut secara jujur di `experiments.json`.

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---

## 1. Mengumpulkan Hasil Notebook 01–05

Notebook 06 tidak mengulang eksperimen; ia **membaca** hasilnya. Kontrak nama
file ada di `_SPEC_BERSAMA.md` bagian 2 dan harus persis:

| Notebook | File yang dibaca |
|---|---|
| 01 — Justifikasi Rasio Split | `json/hasil_split_ratio.json` |
| 02 — Justifikasi Pemilihan `k` KNN | `json/hasil_pemilihan_k.json` |
| 03 — Justifikasi Hyperplane SVM | `json/hasil_svm_hyperplane.json` |
| 04 — Validasi Statistik & Threshold | `json/hasil_validasi_statistik.json` |
| 05 — Ablation, Robustness & Keputusan Model | `json/hasil_ablation_robustness.json` |
| 00 — Setup & Reproduksi Baseline | `json/hasil_baseline_v2.json` |

Kalau sebuah file tidak ditemukan, notebook **tidak berhenti**. Ia mencetak
peringatan dan memakai `NILAI_CADANGAN`, yaitu angka baseline V2 yang sudah
terverifikasi. Setiap bagian `experiments.json` akan diberi label sumbernya
(`notebook 0X` atau `NILAI_CADANGAN (V2)`) supaya tidak ada angka yang
tampil di website tanpa jejak asal-usul.

In [ ]:
# ============================================================
# CELL 7: Muat Hasil Notebook 01-05 + Nilai Cadangan
# ============================================================
def muat_hasil(nama):
    """Baca {OUTPUT_DIR}/json/{nama}.json.

    Kembalikan dict/list bila berhasil, atau None + PERINGATAN bila gagal.
    Notebook sengaja TIDAK crash supaya tetap bisa menghasilkan artefak
    produksi walau sebagian eksperimen belum dijalankan.
    """
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'[OK]         {nama:<28} <- {path}')
        return data
    except FileNotFoundError:
        print(f'[PERINGATAN] {nama:<28} TIDAK DITEMUKAN ({path})')
        return None
    except json.JSONDecodeError as e:
        print(f'[PERINGATAN] {nama:<28} file rusak / bukan JSON valid ({e})')
        return None
    except Exception as e:
        print(f'[PERINGATAN] {nama:<28} gagal dibaca ({type(e).__name__}: {e})')
        return None


garis('MEMUAT HASIL NOTEBOOK 00-05')
H00 = muat_hasil('hasil_baseline_v2')          # notebook 00
H01 = muat_hasil('hasil_split_ratio')          # notebook 01
H02 = muat_hasil('hasil_pemilihan_k')          # notebook 02
H03 = muat_hasil('hasil_svm_hyperplane')       # notebook 03
H04 = muat_hasil('hasil_validasi_statistik')   # notebook 04
H05 = muat_hasil('hasil_ablation_robustness')  # notebook 05
print()

# ------------------------------------------------------------------
# NILAI_CADANGAN: angka baseline V2 yang sudah terverifikasi.
# Dipakai HANYA bila file JSON notebook terkait belum tersedia, supaya
# notebook 06 tetap menghasilkan experiments.json yang lengkap.
# Semua angka di bawah ini berasal dari output notebook V2
# (Diabetes_Prediction_RF_KNN_SVM_V2.ipynb), bukan angka karangan.
# ------------------------------------------------------------------
NILAI_CADANGAN = {
    'dataset': {
        'sumber': 'Kaggle - iammustafatz/diabetes-prediction-dataset',
        'baris_mentah': 100000,
        'duplikat_dihapus': 3854,
        'baris_dipakai': 96146,
        'n_sehat': 87664,
        'n_diabetes': 8482,
        'persen_positif': 8.82,
        'rasio_imbalanced': '10.3:1',
        'fitur': ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level'],
    },
    'perbandingan_model': [
        {'model': 'Random Forest', 'threshold': 0.4965,
         'accuracy': 0.8933, 'precision': 0.4481, 'recall': 0.9057, 'f1': 0.5995,
         'roc_auc': 0.9733, 'ap_score': 0.8695,
         'accuracy_default': 0.8943, 'precision_default': 0.4505,
         'recall_default': 0.9021, 'f1_default': 0.6009,
         'recall_cv_mean': 0.9024, 'recall_cv_std': 0.0118, 'overfit_gap': 0.0126,
         'waktu_latih_s': 5132.6, 'waktu_infer_ms': 353.0},
        {'model': 'KNN', 'threshold': 0.3810,
         'accuracy': 0.8559, 'precision': 0.3710, 'recall': 0.9121, 'f1': 0.5274,
         'roc_auc': 0.9524, 'ap_score': 0.7976,
         'accuracy_default': 0.8877, 'precision_default': 0.4318,
         'recall_default': 0.8644, 'f1_default': 0.5759,
         'recall_cv_mean': 0.8803, 'recall_cv_std': 0.0064, 'overfit_gap': 0.0853,
         'waktu_latih_s': 539.1, 'waktu_infer_ms': 1550.0},
        {'model': 'SVM (Linear)', 'threshold': 0.4951,
         'accuracy': 0.8775, 'precision': 0.4097, 'recall': 0.8833, 'f1': 0.5598,
         'roc_auc': 0.9581, 'ap_score': 0.8063,
         'accuracy_default': 0.8790, 'precision_default': 0.4127,
         'recall_default': 0.8797, 'f1_default': 0.5619,
         'recall_cv_mean': 0.8843, 'recall_cv_std': 0.0113, 'overfit_gap': -0.0003,
         'waktu_latih_s': 32.1, 'waktu_infer_ms': 7.0},
    ],
    'justifikasi_split': {
        'rasio_terpilih': 0.20,
        'n_latih': 76916,
        'n_uji': 19230,
        'n_positif_uji': 1696,
        'margin_of_error_recall_pp': 1.39,
        'alasan': ('Rasio 80:20 stratified menyisakan 1.696 kasus diabetes di test set. '
                   'Dengan recall sekitar 0,906 margin of error 95% (Wald) hanya sekitar '
                   '+/-1,4 poin persen, cukup sempit untuk klaim skripsi, sementara data '
                   'latih tetap 76.916 baris.'),
        'tabel': [],
        'learning_curve': {},
        'catatan': 'Eksperimen rasio lengkap ada di notebook 01. Jalankan notebook 01 untuk mengisi tabel & learning curve.',
    },
    'justifikasi_k': {
        'k_terpilih': 21,
        'recall_cv': 0.8803,
        'alasan': ('k=21 adalah hasil RandomizedSearchCV (5-fold, scoring recall) pada notebook V2. '
                   'Nilai k ganjil menghindari hasil seri pada klasifikasi biner, dan k besar '
                   'meredam noise akibat SMOTE.'),
        'sweep': [],
        'one_se_rule': {},
        'grid_weights_metric': [],
        'catatan': 'Sweep k dan one-SE rule ada di notebook 02.',
    },
    'justifikasi_svm': {
        'kernel_terpilih': 'linear',
        'C_terpilih': 0.1,
        'recall_cv': 0.8843,
        'alasan': ('Kernel linear dengan C=0.1 dipilih lewat RandomizedSearchCV. C kecil berarti '
                   'margin lebar (regularisasi kuat), cocok karena kelas sudah dipisahkan dengan '
                   'baik oleh HbA1c dan kadar glukosa.'),
        'perbandingan_kernel': [],
        'analisis_margin': [],
        'bobot_w': {},
        'catatan': 'Perbandingan kernel, analisis margin, dan visualisasi hyperplane ada di notebook 03.',
    },
    'validasi_statistik': {
        'uji_statistik': [
            {'perbandingan': 'RF vs KNN', 'uji': "McNemar", 'b': 1076, 'c': 356,
             'chi2': 361.0063, 'p_value': 1.700079e-80, 'signifikan': True},
            {'perbandingan': 'RF vs SVM', 'uji': "McNemar", 'b': 915, 'c': 611,
             'chi2': 60.1632, 'p_value': 8.731066e-15, 'signifikan': True},
            {'perbandingan': 'KNN vs SVM', 'uji': "McNemar", 'b': 775, 'c': 1191,
             'chi2': 87.6017, 'p_value': 8.005462e-21, 'signifikan': True},
        ],
        'repeated_cv': [
            {'model': 'Random Forest', 'recall_mean': 0.9024, 'recall_std': 0.0118,
             'train_recall': 0.9151, 'overfit_gap': 0.0126, 'skema': '5-fold stratified'},
            {'model': 'KNN', 'recall_mean': 0.8803, 'recall_std': 0.0064,
             'train_recall': 0.9657, 'overfit_gap': 0.0853, 'skema': '5-fold stratified'},
            {'model': 'SVM (Linear)', 'recall_mean': 0.8843, 'recall_std': 0.0113,
             'train_recall': 0.8840, 'overfit_gap': -0.0003, 'skema': '5-fold stratified'},
        ],
        'nested_cv': [],
        'strategi_threshold': [
            {'model': 'Random Forest', 'metode': "Youden's J", 'threshold': 0.4965},
            {'model': 'KNN', 'metode': "Youden's J", 'threshold': 0.3810},
            {'model': 'SVM (Linear)', 'metode': "Youden's J", 'threshold': 0.4951},
        ],
        'kalibrasi': [],
        'catatan': 'Repeated CV, nested CV, dan analisis kalibrasi lengkap ada di notebook 04.',
    },
    'ablation': {
        'resampling': [],
        'fitur': [],
        'robustness': [],
        'subgrup': [],
        'model_produksi': 'Random Forest',
        'catatan': 'Ablation resampling/fitur, uji robustness, dan analisis subgrup ada di notebook 05.',
    },
    'matriks_keputusan': [
        {'model': 'Random Forest', 'recall': 0.9057, 'precision': 0.4481, 'roc_auc': 0.9733,
         'stabilitas_cv_std': 0.0118, 'overfit_gap': 0.0126, 'waktu_infer_ms': 353.0,
         'interpretabilitas': 'Tinggi (feature importance + TreeSHAP)', 'peringkat': 1,
         'catatan': 'Recall hampir setara KNN tetapi presisi, ROC-AUC, dan kestabilan CV paling baik.'},
        {'model': 'SVM (Linear)', 'recall': 0.8833, 'precision': 0.4097, 'roc_auc': 0.9581,
         'stabilitas_cv_std': 0.0113, 'overfit_gap': -0.0003, 'waktu_infer_ms': 7.0,
         'interpretabilitas': 'Sedang (bobot w linier)', 'peringkat': 2,
         'catatan': 'Paling ringan dan tidak overfit, tetapi recall paling rendah.'},
        {'model': 'KNN', 'recall': 0.9121, 'precision': 0.3710, 'roc_auc': 0.9524,
         'stabilitas_cv_std': 0.0064, 'overfit_gap': 0.0853, 'waktu_infer_ms': 1550.0,
         'interpretabilitas': 'Rendah (lazy learner)', 'peringkat': 3,
         'catatan': 'Recall tertinggi, tetapi presisi terendah, overfit gap 0,085, dan inferensi paling lambat sehingga tidak layak produksi.'},
    ],
    'xai': {
        'metode': 'Permutation Importance (Mean Decrease Recall)',
        'model_sumber': 'KNN',
        'label_di_v2': 'SHAP (label ini KELIRU pada notebook V2)',
        'ranking': [
            {'fitur': 'HbA1c_level', 'label': 'HbA1c', 'nilai': 0.3571},
            {'fitur': 'blood_glucose_level', 'label': 'Kadar Glukosa', 'nilai': 0.2048},
            {'fitur': 'age', 'label': 'Usia', 'nilai': 0.1286},
            {'fitur': 'bmi', 'label': 'BMI', 'nilai': 0.0667},
            {'fitur': 'hypertension', 'label': 'Hipertensi', 'nilai': 0.0238},
        ],
    },
}

# ------------------------------------------------------------------
# Tabel status: apa yang terbaca, apa yang jatuh ke nilai cadangan
# ------------------------------------------------------------------
def ringkas_hasil(nama, obj):
    """Ringkasan satu baris untuk tabel status."""
    if obj is None:
        return 'Tidak ada file -> memakai NILAI_CADANGAN'
    try:
        if nama == 'hasil_split_ratio':
            return f"rasio_terpilih = {obj.get('rasio_terpilih')}"
        if nama == 'hasil_pemilihan_k':
            return f"k_terpilih = {obj.get('k_terpilih')}"
        if nama == 'hasil_svm_hyperplane':
            return f"kernel = {obj.get('kernel_terpilih')}, C = {obj.get('C_terpilih')}"
        if nama == 'hasil_validasi_statistik':
            return f"{len(obj.get('uji_statistik', []))} uji statistik, {len(obj.get('repeated_cv', []))} baris repeated CV"
        if nama == 'hasil_ablation_robustness':
            return f"model_produksi = {obj.get('model_produksi')}, {len(obj.get('matriks_keputusan', []))} baris matriks keputusan"
        if nama == 'hasil_baseline_v2':
            return f"{len(obj.get('perbandingan_model', []))} model baseline terbaca"
    except Exception:
        return '(struktur JSON tidak dikenali)'
    return '(terbaca)'


DAFTAR_INPUT = [
    ('00', 'hasil_baseline_v2', H00),
    ('01', 'hasil_split_ratio', H01),
    ('02', 'hasil_pemilihan_k', H02),
    ('03', 'hasil_svm_hyperplane', H03),
    ('04', 'hasil_validasi_statistik', H04),
    ('05', 'hasil_ablation_robustness', H05),
]

df_status = pd.DataFrame([{
    'Notebook'  : nb,
    'Nama Hasil': nama,
    'Ditemukan' : 'Ya' if obj is not None else 'Tidak',
    'Ringkasan' : ringkas_hasil(nama, obj),
} for nb, nama, obj in DAFTAR_INPUT])

garis('STATUS INPUT NOTEBOOK 06')
simpan_tabel(df_status, 'final_status_input')

n_ada = int((df_status['Ditemukan'] == 'Ya').sum())
print()
print(f'Hasil terbaca : {n_ada} dari {len(DAFTAR_INPUT)} file.')
if n_ada < len(DAFTAR_INPUT):
    print()
    print('PERHATIAN')
    print('  Sebagian hasil eksperimen belum terbaca, sehingga bagian tersebut')
    print('  akan diisi memakai NILAI_CADANGAN (angka baseline V2).')
    print('  Jalankan notebook 00-05 dengan PAKAI_DRIVE=True agar hasil terbaca di sini.')
    print('  Urutan yang benar: 00 -> 01 -> 02 -> 03 -> 04 -> 05 -> 06,')
    print('  semuanya memakai OUTPUT_DIR yang sama.')
else:
    print('Semua hasil eksperimen terbaca. experiments.json akan memakai angka V3 sepenuhnya.')

---

## 2. Konfigurasi Final yang Terjustifikasi

Inti dari revisi ini: **tidak boleh ada angka yang dipakai tanpa alasan.**
Cell berikut mengambil setiap keputusan desain dari notebook yang membuktikannya,
lalu mencetak tabel "Keputusan Desain & Sumber Justifikasinya".

Tabel itu bisa langsung disalin ke Bab 3 (Metodologi) atau dipakai sebagai slide
saat sidang — ketika penguji bertanya "kenapa angkanya segini?", jawabannya ada
di kolom *Notebook Sumber*.

In [ ]:
# ============================================================
# CELL 8: Konfigurasi Final yang Terjustifikasi
# ============================================================
def ambil(obj, kunci, default):
    """Ambil nilai dari hasil notebook; pakai default bila belum tersedia."""
    if isinstance(obj, dict) and obj.get(kunci) is not None:
        return obj[kunci], True
    return default, False


rasio_terpilih,  src_rasio  = ambil(H01, 'rasio_terpilih',   0.20)
k_terpilih,      src_k      = ambil(H02, 'k_terpilih',       21)
kernel_terpilih, src_kernel = ambil(H03, 'kernel_terpilih',  'linear')
C_terpilih,      src_C      = ambil(H03, 'C_terpilih',       0.1)
model_produksi,  src_model  = ambil(H05, 'model_produksi',   'Random Forest')

rasio_terpilih  = float(rasio_terpilih)
k_terpilih      = int(k_terpilih)
kernel_terpilih = str(kernel_terpilih)
C_terpilih      = float(C_terpilih)
model_produksi  = str(model_produksi)

# Hyperparameter model final = hasil tuning notebook V2 (PARAM_RF_V2 di CELL 5).
# Ini yang membedakan model final dari model produksi lama: train_model.py memakai
# RandomForestClassifier(n_estimators=100) tanpa max_depth, sehingga pohon tumbuh
# sampai daun murni dan pickle-nya membengkak sekitar 78 MB.
PARAM_RF_FINAL = dict(PARAM_RF_V2)

def sumber(dipakai, nb):
    return f'Notebook {nb}' if dipakai else f'Notebook {nb} (belum ada -> nilai cadangan)'

keputusan = [
    {'Keputusan': 'Rasio split data',
     'Nilai': f'{int((1-rasio_terpilih)*100)}:{int(rasio_terpilih*100)} (stratified, seed 42)',
     'Notebook Sumber': sumber(src_rasio, '01'),
     'Alasan Ringkas': 'Test set menyisakan cukup kasus positif sehingga margin of error recall tetap sempit, tanpa mengorbankan ukuran data latih.'},
    {'Keputusan': 'Nilai k untuk KNN',
     'Nilai': f'k = {k_terpilih}',
     'Notebook Sumber': sumber(src_k, '02'),
     'Alasan Ringkas': 'Hasil sweep k + one-SE rule; k ganjil menghindari hasil seri, k besar meredam noise dari SMOTE.'},
    {'Keputusan': 'Kernel SVM',
     'Nilai': f'kernel = {kernel_terpilih}',
     'Notebook Sumber': sumber(src_kernel, '03'),
     'Alasan Ringkas': 'Kernel non-linear tidak memberi tambahan recall yang berarti, sedangkan kernel linear jauh lebih murah dan bobotnya bisa dibaca.'},
    {'Keputusan': 'Parameter C SVM',
     'Nilai': f'C = {C_terpilih}',
     'Notebook Sumber': sumber(src_C, '03'),
     'Alasan Ringkas': 'C kecil berarti margin lebar (regularisasi kuat), stabil pada data yang sudah terpisah baik oleh HbA1c dan glukosa.'},
    {'Keputusan': 'Model untuk produksi',
     'Nilai': model_produksi,
     'Notebook Sumber': sumber(src_model, '05'),
     'Alasan Ringkas': 'Matriks keputusan multi-kriteria: recall tinggi, presisi & ROC-AUC terbaik, overfit gap kecil, dan mendukung XAI berbasis pohon.'},
    {'Keputusan': 'Penanganan imbalance',
     'Nilai': 'SMOTE di dalam ImbPipeline',
     'Notebook Sumber': 'Notebook 05 (ablation resampling)',
     'Alasan Ringkas': 'SMOTE hanya aktif saat fit, tidak pada data validasi/uji, sehingga tidak terjadi data leakage.'},
    {'Keputusan': 'Threshold keputusan',
     'Nilai': "Youden's J (dihitung ulang di CELL 9)",
     'Notebook Sumber': 'Notebook 04 (strategi threshold)',
     'Alasan Ringkas': 'Konteks skrining medis mementingkan recall; Youden memaksimalkan (sensitivitas + spesifisitas - 1).'},
    {'Keputusan': 'Hyperparameter Random Forest',
     'Nilai': ', '.join(f'{k}={v}' for k, v in PARAM_RF_FINAL.items()),
     'Notebook Sumber': 'Tuning V2 (RandomizedSearchCV, scoring recall)',
     'Alasan Ringkas': 'max_depth=10 dan min_samples_leaf=4 menahan pertumbuhan pohon: model lebih kecil, lebih cepat, dan overfit gap tetap di bawah 0,05.'},
    {'Keputusan': 'Urutan fitur input',
     'Nilai': ' -> '.join(SELECTED_FEATURES),
     'Notebook Sumber': 'Kontrak SPEC BERSAMA + CELL 10 (uji regresi)',
     'Alasan Ringkas': 'Urutan ini dikunci di model_metadata.json agar bug tertukarnya BMI dan Hipertensi pada inferensi lama tidak terulang.'},
]

df_keputusan = pd.DataFrame(keputusan)
garis('KEPUTUSAN DESAIN & SUMBER JUSTIFIKASINYA')
simpan_tabel(df_keputusan, 'final_keputusan_desain')

print()
print('Konfigurasi final yang dipakai untuk melatih model produksi:')
print(f'  Rasio test set  : {rasio_terpilih:.2f}')
print(f'  Model produksi  : {model_produksi}')
print(f'  Hyperparameter  : {PARAM_RF_FINAL}')
print(f'  Urutan fitur    : {SELECTED_FEATURES}')

---

## 3. Melatih Model Final

### Kenapa scaler dan classifier diekspor terpisah?

Website memanggil `model/predict.py`, dan isi kodenya kurang lebih:

```python
scaler = pickle.load(open('scaler.pkl', 'rb'))
rf     = pickle.load(open('rf_model.pkl', 'rb'))
input_scaled = scaler.transform(input_data)
prob = rf.predict_proba(input_scaled)[0][1]
```

Artinya produksi butuh **dua objek terpisah**, bukan satu objek `Pipeline`.
Maka model final tetap dilatih sebagai `ImbPipeline` (supaya bebas data leakage
dan identik dengan notebook 01–05), lalu komponennya diambil kembali:

```
pipa_final.named_steps['scaler']  ->  scaler.pkl
pipa_final.named_steps['clf']     ->  rf_model.pkl
```

### Kenapa urutan `StandardScaler -> SMOTE -> RF` itu penting di sini?

Karena scaler berada **sebelum** SMOTE, `scaler.fit` hanya melihat **data latih
asli** — bukan sampel sintetis buatan SMOTE. Jadi `mean_` dan `scale_` yang
diekspor mewakili distribusi pasien nyata, persis seperti data yang akan
ditemui saat inferensi. Kalau urutannya dibalik (SMOTE dulu, baru scaler),
statistik scaler akan tercemar sampel sintetis dan hasil prediksi produksi
akan bergeser dari hasil di notebook.

Cell di bawah juga membuktikan secara numerik bahwa
`rf.predict_proba(scaler.transform(X))` menghasilkan probabilitas yang sama
dengan `pipeline.predict_proba(X)` — jadi jalur inferensi produksi setara
dengan jalur evaluasi di notebook.

In [ ]:
# ============================================================
# CELL 9: Latih Model Final + Evaluasi Test Set
# ============================================================
garis('PELATIHAN MODEL FINAL')

# --- 1) Split memakai rasio hasil notebook 01 --------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all[SELECTED_FEATURES], y_all,
    test_size=rasio_terpilih, random_state=RANDOM_STATE, stratify=y_all
)

# CATATAN PRODUKSI (penting):
# predict.py mengirim list biasa ([[...]]) tanpa nama kolom. Karena itu model
# final dilatih memakai numpy array, bukan DataFrame, supaya scaler dan
# classifier tidak menyimpan atribut feature_names_in_. Tanpa ini, setiap
# panggilan inferensi akan memunculkan peringatan "X does not have valid
# feature names". Urutan kolom tetap dikunci oleh SELECTED_FEATURES.
X_train_np = X_train[SELECTED_FEATURES].to_numpy(dtype=float)
X_test_np  = X_test[SELECTED_FEATURES].to_numpy(dtype=float)
y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()

print(f'Data latih : {len(X_train_np):,} baris ({int(y_train_np.sum()):,} diabetes)')
print(f'Data uji   : {len(X_test_np):,} baris ({int(y_test_np.sum()):,} diabetes)')
print(f'Urutan kolom yang dilatih: {SELECTED_FEATURES}')
print()

# --- 2) Latih pipeline final -------------------------------------------------
# Urutan: StandardScaler -> SMOTE -> RandomForest.
# Scaler berada SEBELUM SMOTE, sehingga scaler ter-fit pada data latih ASLI
# (bukan pada sampel sintetis SMOTE). Ini yang membuat scaler.pkl konsisten
# dengan data nyata yang masuk saat inferensi di website.
pipa_final = buat_pipeline_rf(pakai_smote=True)
print('Melatih ImbPipeline final (StandardScaler -> SMOTE -> RandomForest)...')
t0 = time.time()
pipa_final.fit(X_train_np, y_train_np)
waktu_latih_final = time.time() - t0
print(f'Selesai dalam {waktu_latih_final:.1f} detik')
print()

# --- 3) Ambil komponen terpisah untuk produksi -------------------------------
scaler_produksi = pipa_final.named_steps['scaler']
rf_produksi     = pipa_final.named_steps['clf']

print('Komponen yang akan diekspor:')
print(f'  scaler.pkl    : {type(scaler_produksi).__name__} '
      f'(ter-fit pada {len(X_train_np):,} baris data latih ASLI, tanpa SMOTE)')
print(f'  rf_model.pkl  : {type(rf_produksi).__name__} '
      f'({rf_produksi.n_estimators} pohon, max_depth={rf_produksi.max_depth})')
print()

# --- 4) Bukti bahwa jalur produksi setara dengan jalur pipeline --------------
t0 = time.time()
proba_pipa = pipa_final.predict_proba(X_test_np)[:, 1]
waktu_infer_final = (time.time() - t0) * 1000

# Jalur yang dipakai predict.py: scaler.transform() lalu model.predict_proba()
proba_produksi = rf_produksi.predict_proba(scaler_produksi.transform(X_test_np))[:, 1]
selisih_maks = float(np.max(np.abs(proba_pipa - proba_produksi)))

garis('VERIFIKASI JALUR INFERENSI PRODUKSI')
print('pipeline.predict_proba(X)  vs  rf.predict_proba(scaler.transform(X))')
print(f'  Selisih probabilitas maksimum : {selisih_maks:.12f}')
print(f'  Status                        : '
      f'{"SETARA (aman diekspor terpisah)" if selisih_maks < 1e-9 else "TIDAK SETARA - PERIKSA PIPELINE"}')
print()

# --- 5) Threshold Youden + metrik final --------------------------------------
threshold_final = threshold_youden(y_test_np, proba_pipa)
y_pred_default  = (proba_pipa >= 0.5).astype(int)
y_pred_final    = (proba_pipa >= threshold_final).astype(int)

metrik_default = hitung_metrik(y_test_np, y_pred_default, proba_pipa)
metrik_final   = hitung_metrik(y_test_np, y_pred_final,   proba_pipa)

cm = confusion_matrix(y_test_np, y_pred_final)
tn, fp, fn, tp = [int(v) for v in cm.ravel()]
spesifisitas = tn / (tn + fp) if (tn + fp) else 0.0
lo, hi, moe = ci95_proporsi(metrik_final['recall'], int(y_test_np.sum()))

df_metrik_final = pd.DataFrame([
    {'Metrik': 'Threshold',   'Default (0.5)': 0.5,
     f'Youden ({threshold_final:.4f})': threshold_final},
    *[{'Metrik': nama,
       'Default (0.5)': metrik_default[kunci],
       f'Youden ({threshold_final:.4f})': metrik_final[kunci]}
      for nama, kunci in [('Accuracy', 'accuracy'), ('Precision', 'precision'),
                          ('Recall (utama)', 'recall'), ('F1-Score', 'f1'),
                          ('ROC-AUC', 'roc_auc'), ('AP Score', 'ap_score'),
                          ('Brier Score', 'brier')]],
]).round(4)

garis('METRIK MODEL FINAL PADA TEST SET')
simpan_tabel(df_metrik_final, 'final_metrik_model')

print()
print(f'Confusion matrix (threshold {threshold_final:.4f}):')
print(f'  True Negative  : {tn:,}   False Positive : {fp:,}')
print(f'  False Negative : {fn:,}   True Positive  : {tp:,}')
print(f'  Spesifisitas   : {spesifisitas:.4f}')
print(f'  Recall 95% CI  : [{lo:.4f}, {hi:.4f}]  (margin of error +/-{moe*100:.2f} poin persen)')
print(f'  Waktu inferensi: {waktu_infer_final:.1f} ms untuk {len(X_test_np):,} baris '
      f'({waktu_infer_final/len(X_test_np):.4f} ms per sampel)')

# --- 6) Gambar: confusion matrix + ROC + PR ----------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Prediksi Sehat', 'Prediksi Diabetes'],
            yticklabels=['Aktual Sehat', 'Aktual Diabetes'])
axes[0].set_title(f'Confusion Matrix (threshold {threshold_final:.4f})')
axes[0].grid(False)

fpr, tpr, _ = roc_curve(y_test_np, proba_pipa)
axes[1].plot(fpr, tpr, color=WARNA_MODEL['Random Forest'], lw=2,
             label=f"ROC-AUC = {metrik_final['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Tebakan acak')
axes[1].scatter([fp / (fp + tn)], [tp / (tp + fn)], color=WARNA_AKSEN, s=90,
                zorder=5, label='Titik operasi (Youden)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Kurva ROC - Model Final')
axes[1].legend(loc='lower right')

prec, rec, _ = precision_recall_curve(y_test_np, proba_pipa)
axes[2].plot(rec, prec, color=WARNA_MODEL['Random Forest'], lw=2,
             label=f"AP = {metrik_final['ap_score']:.4f}")
axes[2].axhline(float(y_test_np.mean()), color='gray', ls='--', lw=1,
                label=f'Prevalensi = {y_test_np.mean():.4f}')
axes[2].scatter([metrik_final['recall']], [metrik_final['precision']],
                color=WARNA_AKSEN, s=90, zorder=5, label='Titik operasi (Youden)')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Kurva Precision-Recall - Model Final')
axes[2].legend(loc='upper right')

plt.suptitle('Evaluasi Model Final Random Forest pada Test Set', fontsize=14, y=1.02)
plt.tight_layout()
simpan_gambar('final_evaluasi_model')
plt.show()

# --- 7) Simpan ringkasan model final -----------------------------------------
hasil_model_final = {
    'model': model_produksi,
    'hyperparameter': PARAM_RF_FINAL,
    'rasio_uji': rasio_terpilih,
    'n_latih': int(len(X_train_np)),
    'n_uji': int(len(X_test_np)),
    'threshold': threshold_final,
    'metrik_default': metrik_default,
    'metrik_youden': metrik_final,
    'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
    'spesifisitas': spesifisitas,
    'recall_ci95': [float(lo), float(hi)],
    'waktu_latih_s': waktu_latih_final,
    'waktu_infer_ms': waktu_infer_final,
    'selisih_maks_pipeline_vs_produksi': selisih_maks,
}
simpan_json(hasil_model_final, 'hasil_model_final')

---

## 4. Uji Regresi: Konsistensi Urutan Fitur

Ini bagian yang mendokumentasikan bug produksi.

Model dilatih dengan urutan kolom
`['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']`,
tetapi `model/predict.py` versi lama menyusun input sebagai
`[age, hypertension, bmi, HbA1c_level, blood_glucose_level]`.

Akibatnya, setiap prediksi di website memasukkan **nilai hipertensi (0 atau 1)
ke slot BMI**, dan **nilai BMI (sekitar 20–40) ke slot hipertensi**. Model
tidak error — array-nya tetap berukuran 5 — sehingga bug ini tidak terlihat
sampai hasilnya diperiksa satu per satu. Inilah kelas bug yang paling berbahaya:
diam, tapi salah.

Cell berikut melakukan dua hal:

1. Memprediksi tiga pasien contoh (risiko rendah / sedang / tinggi) dengan
   urutan yang **benar** — angka ini menjadi acuan verifikasi setelah deploy.
2. Mengulang prediksi yang sama dengan urutan yang **tertukar** (bug lama),
   lalu mencetak selisih probabilitasnya sebagai bukti bahwa bug tersebut
   material, bukan sekadar catatan kosmetik.

In [ ]:
# ============================================================
# CELL 10: Verifikasi Konsistensi Urutan Fitur (Uji Regresi Bug)
# ============================================================
garis('UJI REGRESI: URUTAN FITUR')

# Tiga pasien contoh. Sengaja ditulis sebagai dict BERNAMA supaya urutan
# penulisan tidak menentukan apa pun - array disusun berdasarkan
# SELECTED_FEATURES, bukan berdasarkan urutan kunci di dict ini.
PASIEN_CONTOH = {
    'Risiko rendah': {
        'age': 25.0, 'bmi': 22.0, 'hypertension': 0,
        'HbA1c_level': 5.0, 'blood_glucose_level': 90.0,
    },
    'Risiko sedang': {
        'age': 48.0, 'bmi': 29.0, 'hypertension': 0,
        'HbA1c_level': 6.2, 'blood_glucose_level': 145.0,
    },
    'Risiko tinggi': {
        'age': 62.0, 'bmi': 34.5, 'hypertension': 1,
        'HbA1c_level': 7.8, 'blood_glucose_level': 210.0,
    },
}

# Urutan yang dipakai kode inferensi LAMA (salah): bmi dan hypertension tertukar
URUTAN_BUG = ['age', 'hypertension', 'bmi', 'HbA1c_level', 'blood_glucose_level']

def prediksi_produksi(vektor):
    """Meniru persis jalur predict.py: scaler.transform lalu predict_proba."""
    x = np.array([vektor], dtype=float)
    return float(rf_produksi.predict_proba(scaler_produksi.transform(x))[0, 1])

def label_keputusan(prob):
    return 'DIABETES' if prob >= threshold_final else 'SEHAT'


print(f'Urutan training (BENAR) : {SELECTED_FEATURES}')
print(f'Urutan inferensi lama   : {URUTAN_BUG}   <- bmi & hypertension tertukar')
print(f'Threshold keputusan     : {threshold_final:.4f}')
print()

baris_uji = []
for nama, p in PASIEN_CONTOH.items():
    vektor_benar = [p[f] for f in SELECTED_FEATURES]
    vektor_bug   = [p[f] for f in URUTAN_BUG]

    prob_benar = prediksi_produksi(vektor_benar)
    prob_bug   = prediksi_produksi(vektor_bug)

    baris_uji.append({
        'Pasien': nama,
        'Usia': p['age'],
        'BMI': p['bmi'],
        'Hipertensi': p['hypertension'],
        'HbA1c': p['HbA1c_level'],
        'Glukosa': p['blood_glucose_level'],
        'Prob (urutan benar)': round(prob_benar, 4),
        'Keputusan (benar)': label_keputusan(prob_benar),
        'Prob (urutan bug)': round(prob_bug, 4),
        'Keputusan (bug)': label_keputusan(prob_bug),
        'Selisih Prob': round(prob_bug - prob_benar, 4),
        'Keputusan Berubah': 'YA' if label_keputusan(prob_bug) != label_keputusan(prob_benar) else 'tidak',
    })

df_uji_urutan = pd.DataFrame(baris_uji)
simpan_tabel(df_uji_urutan, 'final_uji_urutan_fitur')

print()
garis('HASIL PREDIKSI DENGAN URUTAN YANG BENAR (ACUAN VERIFIKASI DEPLOY)')
for r in baris_uji:
    print(f"  {r['Pasien']:<14} usia={r['Usia']:<5} bmi={r['BMI']:<6} "
          f"hipertensi={r['Hipertensi']} hba1c={r['HbA1c']:<4} glukosa={r['Glukosa']:<6} "
          f"-> prob={r['Prob (urutan benar)']:.4f} ({r['Keputusan (benar)']})")

print()
garis('DAMPAK BUG URUTAN FITUR (bmi <-> hypertension)')
selisih_abs = [abs(r['Selisih Prob']) for r in baris_uji]
n_berubah = sum(1 for r in baris_uji if r['Keputusan Berubah'] == 'YA')
for r in baris_uji:
    print(f"  {r['Pasien']:<14} benar={r['Prob (urutan benar)']:.4f} ({r['Keputusan (benar)']:<8}) | "
          f"bug={r['Prob (urutan bug)']:.4f} ({r['Keputusan (bug)']:<8}) | "
          f"selisih={r['Selisih Prob']:+.4f} | keputusan berubah: {r['Keputusan Berubah']}")
print()
print(f'  Selisih probabilitas terbesar : {max(selisih_abs):.4f} '
      f'({max(selisih_abs)*100:.2f} poin persen)')
print(f'  Rata-rata selisih absolut     : {float(np.mean(selisih_abs)):.4f}')
print(f'  Keputusan akhir berubah pada  : {n_berubah} dari {len(baris_uji)} pasien contoh')
print()
print('  Kesimpulan: penukaran BMI dan Hipertensi bukan kesalahan kosmetik.')
print('  Nilai hipertensi (0/1) masuk ke slot BMI, dan nilai BMI (20-40) masuk ke')
print('  slot hipertensi, sehingga input yang dilihat model sangat jauh dari rentang')
print('  data latih. Karena itu model_metadata.json WAJIB memuat field feature_order,')
print('  dan predict.py WAJIB menyusun array mengikuti field tersebut.')
print()
print('  Catatan: ketiga pasien contoh sengaja dipilih di dalam batas winsorization')
print('  (lihat CELL 11) sehingga hasilnya tidak terpengaruh proses capping.')

hasil_uji_urutan = {
    'urutan_training': SELECTED_FEATURES,
    'urutan_inferensi_lama': URUTAN_BUG,
    'threshold': threshold_final,
    'pasien': baris_uji,
    'selisih_maks': float(max(selisih_abs)),
    'selisih_rata_rata': float(np.mean(selisih_abs)),
    'jumlah_keputusan_berubah': int(n_berubah),
}
simpan_json(hasil_uji_urutan, 'hasil_uji_urutan_fitur')

---

## 5. Export Artefak Produksi

Tiga file diekspor ke `{OUTPUT_DIR}/produksi/`:

| File | Isi | Dipakai oleh |
|---|---|---|
| `rf_model.pkl` | `RandomForestClassifier` hasil tuning, sudah ter-fit | `predict.py` |
| `scaler.pkl` | `StandardScaler` ter-fit pada data latih asli | `predict.py` |
| `model_metadata.json` | Urutan fitur, threshold, batas winsorization, metrik, versi library | `predict.py` + dokumentasi skripsi |

Catatan teknis:

- **`pickle`, bukan `joblib`.** Sistem produksi memuat model dengan `pickle`
  (`joblib` sempat bermasalah dengan asyncio di Windows), jadi format ekspor
  harus mengikuti.
- **`protocol=4`.** Protokol 4 didukung sejak Python 3.4 dan tidak bergantung
  pada fitur baru protokol 5, sehingga pickle tetap terbaca walau versi Python
  di server berbeda dengan versi Colab.
- **Batas winsorization ikut diekspor.** Notebook melakukan capping IQR sebelum
  melatih model, sehingga model tidak pernah melihat BMI 95 atau glukosa 300.
  Kalau server mengirim nilai ekstrem tanpa capping, model diminta melakukan
  ekstrapolasi di luar rentang latihnya. Batas ini diekspor agar `predict.py`
  bisa menerapkan capping yang sama.

In [ ]:
# ============================================================
# CELL 11: Export Artefak Produksi (pickle + metadata + zip)
# ============================================================
import pickle, shutil, sys, sklearn, imblearn
from datetime import datetime

PRODUKSI_DIR = f'{OUTPUT_DIR}/produksi'
os.makedirs(PRODUKSI_DIR, exist_ok=True)

garis('EXPORT ARTEFAK PRODUKSI')
print(f'Folder tujuan: {PRODUKSI_DIR}')
print()

# --- 1) Pickle model & scaler (protocol=4 agar kompatibel lintas versi) ------
path_model  = f'{PRODUKSI_DIR}/rf_model.pkl'
path_scaler = f'{PRODUKSI_DIR}/scaler.pkl'

with open(path_model, 'wb') as f:
    pickle.dump(rf_produksi, f, protocol=4)
with open(path_scaler, 'wb') as f:
    pickle.dump(scaler_produksi, f, protocol=4)

mb_model  = os.path.getsize(path_model) / (1024 ** 2)
mb_scaler = os.path.getsize(path_scaler) / (1024 ** 2)
print(f'  rf_model.pkl  : {mb_model:.2f} MB')
print(f'  scaler.pkl    : {mb_scaler:.4f} MB')
print(f'  Pembanding    : model produksi lama sekitar 75.2 MB '
      f'(RandomForest default tanpa max_depth)')
if mb_model > 0:
    print(f'  Penyusutan    : sekitar {75.2 / mb_model:.1f}x lebih kecil berkat '
          f'max_depth=10 dan min_samples_leaf=4')
print()

# --- 2) Uji muat ulang: pastikan pickle benar-benar bisa dibaca --------------
with open(path_model, 'rb') as f:
    _cek_model = pickle.load(f)
with open(path_scaler, 'rb') as f:
    _cek_scaler = pickle.load(f)
_x = X_test_np[:200]
_selisih_reload = float(np.max(np.abs(
    _cek_model.predict_proba(_cek_scaler.transform(_x))[:, 1] -
    rf_produksi.predict_proba(scaler_produksi.transform(_x))[:, 1]
)))
print(f'  Uji muat ulang pickle: selisih probabilitas maksimum = {_selisih_reload:.12f}')
print(f'  Status               : {"OK" if _selisih_reload < 1e-12 else "PERIKSA KEMBALI"}')
print()

# --- 3) Batas winsorization --------------------------------------------------
def hitung_batas_winsor(df):
    """Hitung ulang batas capping IQR per fitur.

    Boleh dihitung dari df_clean (data yang sudah di-capping) karena capping
    pada pagar 1,5*IQR tidak mengubah kuartil: nilai di bawah Q1 tetap di bawah
    Q1 setelah dinaikkan ke batas bawah, begitu pula sebaliknya. Jadi Q1, Q3,
    dan IQR-nya identik dengan hasil perhitungan sebelum capping.
    """
    batas = {}
    for feat in SELECTED_FEATURES:
        if df[feat].nunique() > 2:
            Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
            IQR = Q3 - Q1
            batas[feat] = {
                'dicapping': True,
                'batas_bawah': float(Q1 - 1.5 * IQR),
                'batas_atas': float(Q3 + 1.5 * IQR),
            }
        else:
            batas[feat] = {
                'dicapping': False,
                'batas_bawah': float(df[feat].min()),
                'batas_atas': float(df[feat].max()),
            }
    return batas

BATAS_WINSOR = hitung_batas_winsor(df_clean)
print('  Batas winsorization yang diekspor (WAJIB dipakai predict.py):')
for f_, b_ in BATAS_WINSOR.items():
    tanda = 'capping' if b_['dicapping'] else 'biner, tanpa capping'
    print(f'    {f_:<22} [{b_["batas_bawah"]:>8.2f}, {b_["batas_atas"]:>8.2f}]  ({tanda})')
print()

# --- 4) model_metadata.json --------------------------------------------------
model_metadata = {
    'nama_sistem'   : 'DiaPredict',
    'versi_model'   : 'v3.0-revisi',
    'tanggal_dibuat': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dilatih_oleh'  : 'Notebook 06_Model_Final_dan_Export_Produksi.ipynb',
    'algoritma'     : type(rf_produksi).__name__,
    'hyperparameter': PARAM_RF_FINAL,

    # --- KUNCI ANTI-BUG: urutan fitur eksplisit ---
    'feature_order' : list(SELECTED_FEATURES),
    'feature_labels': list(FEATURE_LABELS),
    'feature_order_catatan': (
        'WAJIB. Array input harus disusun persis mengikuti feature_order. '
        'Versi lama predict.py memakai urutan [age, hypertension, bmi, HbA1c_level, '
        'blood_glucose_level] sehingga BMI dan hipertensi tertukar pada setiap '
        'prediksi. Lihat hasil uji regresi di hasil_uji_urutan_fitur.json.'
    ),
    'urutan_argumen_cli_predict_py': ['age', 'hypertension', 'bmi', 'HbA1c_level', 'blood_glucose_level'],
    'urutan_argumen_catatan': (
        'Urutan argumen CLI predict.py dipertahankan agar controller Laravel tidak '
        'perlu diubah, TETAPI script harus menyusun ulang nilainya mengikuti '
        'feature_order sebelum memanggil scaler.transform().'
    ),

    'threshold': {
        'nilai'          : float(threshold_final),
        'metode'         : "Youden's J Statistic (maksimum TPR - FPR pada test set)",
        'threshold_v2'   : 0.4965,
        'catatan'        : 'Threshold dihitung ulang pada model final; ganti nilai 0.4965 di predict.py dengan nilai ini.',
    },

    'preprocessing': {
        'urutan': [
            'hapus duplikat pada dataset penuh',
            'ambil 5 fitur terpilih',
            'winsorization (capping IQR 1.5) pada fitur numerik non-biner',
            'StandardScaler (scaler.pkl)',
            'SMOTE (HANYA saat training, tidak dipakai saat inferensi)',
        ],
        'winsorization': BATAS_WINSOR,
        'scaler': {
            'tipe' : type(scaler_produksi).__name__,
            'mean' : [float(v) for v in scaler_produksi.mean_],
            'scale': [float(v) for v in scaler_produksi.scale_],
            'catatan': ('Scaler ter-fit pada data latih ASLI karena berada sebelum SMOTE '
                        'di dalam pipeline, sehingga statistiknya mewakili distribusi pasien nyata.'),
        },
    },

    'split': {
        'rasio_uji'   : float(rasio_terpilih),
        'rasio_latih' : float(1 - rasio_terpilih),
        'n_latih'     : int(len(X_train_np)),
        'n_uji'       : int(len(X_test_np)),
        'stratified'  : True,
        'random_state': RANDOM_STATE,
        'sumber_justifikasi': 'Notebook 01' if src_rasio else 'Nilai cadangan (baseline V2)',
    },

    'metrik_test_set': {
        'threshold_youden': {k: float(v) for k, v in metrik_final.items()},
        'threshold_default_0_5': {k: float(v) for k, v in metrik_default.items()},
        'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'spesifisitas': float(spesifisitas),
        'recall_ci95': [float(lo), float(hi)],
        'n_uji': int(len(X_test_np)),
    },

    'dataset': {
        'sumber'        : 'Kaggle - iammustafatz/diabetes-prediction-dataset',
        'baris_dipakai' : int(len(df_clean)),
        'n_sehat'       : int((df_clean[TARGET] == 0).sum()),
        'n_diabetes'    : int((df_clean[TARGET] == 1).sum()),
        'persen_positif': float(df_clean[TARGET].mean() * 100),
    },

    'ukuran_file_mb': {
        'rf_model.pkl': round(mb_model, 4),
        'scaler.pkl'  : round(mb_scaler, 6),
        'catatan'     : 'Model lama sekitar 75.2 MB karena RandomForest default tanpa max_depth.',
    },

    'versi_library': {
        'python'          : sys.version.split()[0],
        'scikit_learn'    : sklearn.__version__,
        'imbalanced_learn': imblearn.__version__,
        'numpy'           : np.__version__,
        'pandas'          : pd.__version__,
    },
    'catatan_kompatibilitas': (
        'Pickle disimpan dengan protocol=4. Muat dengan pickle.load(). Untuk menghindari '
        'InconsistentVersionWarning, gunakan versi scikit-learn yang sama di server '
        '(lihat versi_library di atas) atau latih ulang lewat notebook ini.'
    ),
}

path_metadata = f'{PRODUKSI_DIR}/model_metadata.json'
with open(path_metadata, 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2, ensure_ascii=False)
print(f'  model_metadata.json disimpan: {path_metadata}')
simpan_json(model_metadata, 'model_metadata')   # salinan di folder json/
print()

# --- 5) Zip semua artefak agar mudah diunduh dari Colab ----------------------
path_zip = shutil.make_archive(f'{OUTPUT_DIR}/artefak_produksi_diapredict', 'zip', PRODUKSI_DIR)
print(f'  Arsip dibuat: {path_zip} ({os.path.getsize(path_zip)/(1024**2):.2f} MB)')
print('  (Arsip akan diperbarui lagi di CELL 14 setelah experiments.json dibuat.)')
print()
print('  Untuk mengunduh dari Colab, jalankan di cell baru:')
print("      from google.colab import files")
print("      files.download('" + path_zip + "')")
print()

# --- 6) Instruksi salin ke repo Laravel --------------------------------------
garis('CARA MEMASANG KE REPO LARAVEL DiaPredict')
print('1. Unduh arsip zip di atas, lalu ekstrak.')
print('2. Backup dulu artefak lama:')
print('      cp model/rf_model.pkl model/rf_model.pkl.bak')
print('      cp model/scaler.pkl   model/scaler.pkl.bak')
print('3. Salin ketiga file ke folder model/ di root repo Laravel:')
print('      rf_model.pkl        -> model/rf_model.pkl')
print('      scaler.pkl          -> model/scaler.pkl')
print('      model_metadata.json -> model/model_metadata.json')
print('4. Perbarui model/predict.py:')
print('      - susun array input mengikuti feature_order dari model_metadata.json')
print('        (BUKAN mengikuti urutan argumen CLI),')
print('      - terapkan capping memakai preprocessing.winsorization,')
print(f'      - ganti threshold 0.4965 menjadi {threshold_final:.4f}.')
print('5. Jalankan uji verifikasi pada CELL 14 dan bandingkan angkanya.')

---

## 6. Membangun `experiments.json` untuk Website

`experiments.json` adalah **satu-satunya sumber angka** bagi halaman
"Metodologi & Pengujian" di website. Tujuannya sederhana: angka yang tampil di
web harus sama persis dengan angka di skripsi, tanpa ada yang ditulis manual
di Blade.

Strukturnya:

```
{
  "meta"               : identitas model final, threshold, urutan fitur, tanggal build
  "perbandingan_model" : RF vs KNN vs SVM (metrik test set)
  "justifikasi_split"  : jawaban "kenapa 80:20"          (notebook 01)
  "justifikasi_k"      : jawaban "kenapa k sekian"       (notebook 02)
  "justifikasi_svm"    : jawaban "kenapa hyperplane ini" (notebook 03)
  "validasi_statistik" : repeated CV, McNemar, threshold, kalibrasi (notebook 04)
  "ablation"           : ablation resampling/fitur, robustness, subgrup (notebook 05)
  "matriks_keputusan"  : dasar pemilihan model produksi  (notebook 05)
  "xai"                : feature importance model final  (notebook 06, CELL 13)
}
```

### Field `sumber_metode_importance` — soal kejujuran metodologis

Notebook V2 memberi judul "SHAP" pada grafik feature importance-nya, padahal
yang benar-benar dihitung adalah **Permutation Importance (Mean Decrease
Recall)** milik model **KNN** — SHAP tidak pernah dijalankan di sana karena
KNN bukan model berbasis pohon. Website ikut menampilkan label "SHAP" tersebut.

Karena itu `experiments.json` memuat field `sumber_metode_importance` yang
menyatakan metode dan model yang benar-benar dipakai. Website memakai field ini
untuk memberi label yang jujur, dan CELL 13 menghitung ulang importance pada
model final agar labelnya sesuai kenyataan.

Setiap bagian juga membawa field `sumber` yang menyatakan apakah isinya berasal
dari notebook V3 atau dari nilai cadangan V2.

In [ ]:
# ============================================================
# CELL 12: Bangun experiments.json untuk Website
# ============================================================
garis('MEMBANGUN experiments.json')

def pilih_bagian(hasil_nb, kunci_cadangan, label_nb):
    """Ambil isi dari hasil notebook bila ada; kalau tidak, pakai NILAI_CADANGAN.

    Selalu menambahkan field 'sumber' supaya website bisa menandai angka
    mana yang berasal dari eksperimen V3 dan mana yang masih baseline V2.
    """
    if isinstance(hasil_nb, dict) and len(hasil_nb) > 0:
        isi = dict(hasil_nb)
        isi['sumber'] = f'Notebook {label_nb} (V3)'
        return isi
    isi = json.loads(json.dumps(NILAI_CADANGAN[kunci_cadangan]))
    isi['sumber'] = 'NILAI_CADANGAN (baseline V2)'
    return isi


# --- perbandingan model: pakai hasil notebook 00 bila ada --------------------
if isinstance(H00, dict) and H00.get('perbandingan_model'):
    perbandingan_model = list(H00['perbandingan_model'])
    sumber_perbandingan = 'Notebook 00 (V3)'
else:
    perbandingan_model = json.loads(json.dumps(NILAI_CADANGAN['perbandingan_model']))
    sumber_perbandingan = 'NILAI_CADANGAN (baseline V2)'

# --- matriks keputusan: milik notebook 05 -----------------------------------
if isinstance(H05, dict) and H05.get('matriks_keputusan'):
    matriks_keputusan = list(H05['matriks_keputusan'])
    sumber_matriks = 'Notebook 05 (V3)'
else:
    matriks_keputusan = json.loads(json.dumps(NILAI_CADANGAN['matriks_keputusan']))
    sumber_matriks = 'NILAI_CADANGAN (baseline V2)'

SUMBER_METODE_IMPORTANCE = (
    'Permutation Importance (Mean Decrease Recall) pada model final Random Forest, '
    'dihitung ulang di notebook 06. Pada notebook V2 grafik importance diberi label '
    '"SHAP", padahal yang dihitung adalah Permutation Importance milik model KNN; '
    'label tersebut diperbaiki di sini.'
)

experiments = {
    'meta': {
        'nama_sistem'    : 'DiaPredict',
        'judul'          : 'Perbandingan Random Forest, KNN, dan SVM untuk Prediksi Risiko Diabetes',
        'versi_data'     : 'V3 (Revisi Pengujian)',
        'versi_model'    : model_metadata['versi_model'],
        'tanggal_build'  : model_metadata['tanggal_dibuat'],
        'dataset'        : {
            'sumber'        : 'Kaggle - iammustafatz/diabetes-prediction-dataset',
            'baris_dipakai' : int(len(df_clean)),
            'n_sehat'       : int((df_clean[TARGET] == 0).sum()),
            'n_diabetes'    : int((df_clean[TARGET] == 1).sum()),
            'persen_positif': round(float(df_clean[TARGET].mean() * 100), 2),
        },
        'model_produksi' : model_produksi,
        'algoritma'      : model_metadata['algoritma'],
        'hyperparameter' : PARAM_RF_FINAL,
        'feature_order'  : list(SELECTED_FEATURES),
        'feature_labels' : list(FEATURE_LABELS),
        'threshold'      : round(float(threshold_final), 4),
        'threshold_metode': "Youden's J Statistic",
        'rasio_split'    : {'latih': round(1 - rasio_terpilih, 2), 'uji': round(rasio_terpilih, 2)},
        'metrik_model_final': {k: round(float(v), 4) for k, v in metrik_final.items()},
        'confusion_matrix_final': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
        'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
        'catatan_perbaikan': [
            'Urutan fitur inferensi dikunci lewat model_metadata.json (feature_order).',
            'Model produksi memakai hyperparameter hasil tuning, bukan RandomForest default.',
            'Batas winsorization diekspor agar preprocessing server sama dengan notebook.',
            'Label metode feature importance dikoreksi (bukan SHAP milik KNN).',
        ],
    },

    'perbandingan_model': perbandingan_model,
    'sumber_perbandingan_model': sumber_perbandingan,

    'justifikasi_split' : pilih_bagian(H01, 'justifikasi_split', '01'),
    'justifikasi_k'     : pilih_bagian(H02, 'justifikasi_k', '02'),
    'justifikasi_svm'   : pilih_bagian(H03, 'justifikasi_svm', '03'),
    'validasi_statistik': pilih_bagian(H04, 'validasi_statistik', '04'),
    'ablation'          : pilih_bagian(H05, 'ablation', '05'),

    'matriks_keputusan' : matriks_keputusan,
    'sumber_matriks_keputusan': sumber_matriks,

    # Bagian xai diisi ulang oleh CELL 13 memakai model final.
    'xai': {
        'status': 'menunggu perhitungan di CELL 13',
        'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
    },
}


def simpan_experiments(obj):
    """Simpan experiments.json ke folder produksi dan folder json (kontrak spec)."""
    path = f'{PRODUKSI_DIR}/experiments.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    simpan_json(obj, 'experiments')   # salinan sesuai kontrak SPEC BERSAMA
    return path


path_experiments = simpan_experiments(experiments)

# Ringkasan struktur + asal-usul tiap bagian
df_sumber = pd.DataFrame([
    {'Bagian': 'perbandingan_model',  'Sumber': sumber_perbandingan},
    {'Bagian': 'justifikasi_split',   'Sumber': experiments['justifikasi_split']['sumber']},
    {'Bagian': 'justifikasi_k',       'Sumber': experiments['justifikasi_k']['sumber']},
    {'Bagian': 'justifikasi_svm',     'Sumber': experiments['justifikasi_svm']['sumber']},
    {'Bagian': 'validasi_statistik',  'Sumber': experiments['validasi_statistik']['sumber']},
    {'Bagian': 'ablation',            'Sumber': experiments['ablation']['sumber']},
    {'Bagian': 'matriks_keputusan',   'Sumber': sumber_matriks},
    {'Bagian': 'xai',                 'Sumber': 'Notebook 06 (dihitung ulang di CELL 13)'},
])
print()
garis('ASAL-USUL SETIAP BAGIAN experiments.json')
simpan_tabel(df_sumber, 'final_sumber_experiments')

print()
print('Kunci tingkat atas experiments.json:')
for k in experiments.keys():
    print(f'  - {k}')
print()
garis('CARA MEMASANG experiments.json KE WEBSITE')
print('1. Salin file berikut ke repo Laravel:')
print(f'      {path_experiments}')
print('      -> public/data/experiments.json')
print('2. Buat folder tujuan bila belum ada: mkdir -p public/data')
print('3. Halaman metodologi membaca file ini, jadi tidak ada lagi angka yang')
print('   ditulis manual di file Blade.')
print('4. Perhatikan field meta.sumber_metode_importance: pakai isinya sebagai')
print('   label pada grafik feature importance supaya penamaannya jujur.')

---

## 7. Menghitung Ulang XAI pada Model Final

Dua metode dihitung di sini:

1. **Permutation Importance** dengan `scoring='recall'` — mengukur seberapa
   besar recall turun ketika nilai satu fitur diacak. Metrik ini dipilih karena
   recall memang metrik utama pada kasus skrining medis, dan hasilnya bisa
   dibandingkan langsung dengan angka Permutation Importance KNN dari V2.
2. **TreeSHAP** (`mean |SHAP value|`) — dijalankan hanya kalau library `shap`
   tersedia, dibungkus `try/except` supaya notebook tetap jalan tanpa library
   tambahan. Berbeda dengan V2, kali ini TreeSHAP memang **absah** dipakai,
   karena model finalnya Random Forest (model berbasis pohon).

Hasilnya masuk ke `experiments['xai']` dengan **nama metode yang sesuai
kenyataan** — inilah perbaikan atas kekeliruan pelabelan di V2.

Validasi domain medis yang diharapkan: HbA1c dan kadar glukosa harus menempati
peringkat teratas. Kalau tidak, ada indikasi model belajar dari pola yang salah.

In [ ]:
# ============================================================
# CELL 13: XAI Model Final (Permutation Importance + TreeSHAP)
# ============================================================
from sklearn.inspection import permutation_importance

garis('XAI MODEL FINAL')

# Permutation importance dihitung pada data uji yang SUDAH di-scale, karena
# rf_produksi menerima input hasil scaler.transform (persis seperti di produksi).
X_test_scaled = scaler_produksi.transform(X_test_np)

print('Menghitung Permutation Importance (scoring=recall, 10 pengulangan)...')
t0 = time.time()
pi = permutation_importance(
    rf_produksi, X_test_scaled, y_test_np,
    scoring='recall', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
print(f'Selesai dalam {time.time() - t0:.1f} detik')
print()

df_pi = pd.DataFrame({
    'Fitur'      : SELECTED_FEATURES,
    'Label'      : FEATURE_LABELS,
    'Mean Decrease Recall': pi.importances_mean,
    'Std'        : pi.importances_std,
}).sort_values('Mean Decrease Recall', ascending=False).reset_index(drop=True)
df_pi['Peringkat'] = range(1, len(df_pi) + 1)
df_pi = df_pi[['Peringkat', 'Fitur', 'Label', 'Mean Decrease Recall', 'Std']].round(4)

garis('PERMUTATION IMPORTANCE - MODEL FINAL (RANDOM FOREST)')
simpan_tabel(df_pi, 'final_permutation_importance')

# --- TreeSHAP (opsional) -----------------------------------------------------
shap_tersedia = False
df_shap = None
try:
    import shap
    shap_tersedia = True
except ImportError:
    print()
    print('[INFO] Library shap tidak tersedia. Bagian TreeSHAP dilewati.')
    print('       Jalankan "!pip install -q shap" di cell baru bila ingin menghitungnya.')

if shap_tersedia:
    print()
    print('Menghitung TreeSHAP (mean |SHAP value|) pada sampel data uji...')
    try:
        n_sampel_shap = min(2000, len(X_test_scaled))
        idx_shap = np.random.RandomState(RANDOM_STATE).choice(
            len(X_test_scaled), size=n_sampel_shap, replace=False)
        X_shap = X_test_scaled[idx_shap]

        t0 = time.time()
        explainer = shap.TreeExplainer(rf_produksi)
        sv = explainer.shap_values(X_shap)
        sv = np.array(sv)

        # Bentuk output shap berbeda antar versi:
        #   versi lama : (n_kelas, n_sampel, n_fitur)
        #   versi baru : (n_sampel, n_fitur, n_kelas)
        if sv.ndim == 3:
            if sv.shape[0] == 2 and sv.shape[-1] == len(SELECTED_FEATURES):
                sv = sv[1]
            else:
                sv = sv[:, :, 1]

        mean_abs_shap = np.abs(sv).mean(axis=0)
        print(f'Selesai dalam {time.time() - t0:.1f} detik '
              f'({n_sampel_shap:,} sampel)')

        df_shap = pd.DataFrame({
            'Fitur': SELECTED_FEATURES,
            'Label': FEATURE_LABELS,
            'Mean |SHAP|': mean_abs_shap,
        }).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
        df_shap['Peringkat'] = range(1, len(df_shap) + 1)
        df_shap = df_shap[['Peringkat', 'Fitur', 'Label', 'Mean |SHAP|']].round(4)

        print()
        garis('TreeSHAP - MODEL FINAL (RANDOM FOREST)')
        simpan_tabel(df_shap, 'final_treeshap_importance')
    except Exception as e:
        shap_tersedia = False
        df_shap = None
        print(f'[PERINGATAN] Perhitungan TreeSHAP gagal ({type(e).__name__}: {e}). '
              f'Bagian ini dilewati.')

# --- Grafik barh -------------------------------------------------------------
n_panel = 2 if df_shap is not None else 1
fig, axes = plt.subplots(1, n_panel, figsize=(8 * n_panel, 5))
if n_panel == 1:
    axes = [axes]

d = df_pi.sort_values('Mean Decrease Recall')
axes[0].barh(d['Label'], d['Mean Decrease Recall'],
             color=WARNA_MODEL['Random Forest'],
             xerr=df_pi.sort_values('Mean Decrease Recall')['Std'],
             error_kw={'ecolor': '#7f8c8d', 'capsize': 3})
axes[0].set_xlabel('Mean Decrease Recall')
axes[0].set_title('Permutation Importance - Random Forest Final')
for i, (v, lbl) in enumerate(zip(d['Mean Decrease Recall'], d['Label'])):
    axes[0].text(v, i, f'  {v:.4f}', va='center', fontsize=10)

if df_shap is not None:
    ds = df_shap.sort_values('Mean |SHAP|')
    axes[1].barh(ds['Label'], ds['Mean |SHAP|'], color=WARNA_AKSEN)
    axes[1].set_xlabel('Mean |SHAP value|')
    axes[1].set_title('TreeSHAP - Random Forest Final')
    for i, v in enumerate(ds['Mean |SHAP|']):
        axes[1].text(v, i, f'  {v:.4f}', va='center', fontsize=10)

plt.suptitle('Feature Importance Model Final (metode dilabeli sesuai perhitungan sebenarnya)',
             fontsize=13, y=1.03)
plt.tight_layout()
simpan_gambar('final_feature_importance')
plt.show()

# --- Masukkan ke experiments.json -------------------------------------------
metode_utama = 'Permutation Importance (Mean Decrease Recall)'
experiments['xai'] = {
    'metode_utama'   : metode_utama,
    'model_sumber'   : f'{model_produksi} (model final produksi)',
    'scoring'        : 'recall',
    'n_repeats'      : 10,
    'data'           : f'Test set {len(X_test_np):,} baris (sudah di-scale seperti di produksi)',
    'sumber_metode_importance': SUMBER_METODE_IMPORTANCE,
    'permutation_importance': [
        {'peringkat': int(r['Peringkat']), 'fitur': r['Fitur'], 'label': r['Label'],
         'nilai': float(r['Mean Decrease Recall']), 'std': float(r['Std'])}
        for _, r in df_pi.iterrows()
    ],
    'treeshap': (
        [{'peringkat': int(r['Peringkat']), 'fitur': r['Fitur'], 'label': r['Label'],
          'nilai': float(r['Mean |SHAP|'])} for _, r in df_shap.iterrows()]
        if df_shap is not None else []
    ),
    'treeshap_tersedia': bool(df_shap is not None),
    'perbandingan_v2': {
        'metode_v2'        : 'Permutation Importance (Mean Decrease Recall)',
        'model_v2'         : 'KNN',
        'label_grafik_v2'  : 'SHAP',
        'catatan'          : ('Grafik V2 diberi label SHAP padahal yang dihitung adalah '
                              'Permutation Importance pada KNN. TreeSHAP tidak dapat dipakai '
                              'pada KNN karena bukan model berbasis pohon. Di V3, TreeSHAP '
                              'dipakai secara sah karena model finalnya Random Forest.'),
        'ranking_v2'       : NILAI_CADANGAN['xai']['ranking'],
    },
}
experiments['meta']['sumber_metode_importance'] = SUMBER_METODE_IMPORTANCE

simpan_experiments(experiments)

print()
garis('VALIDASI DOMAIN MEDIS')
tiga_teratas = list(df_pi['Fitur'].head(3))
print(f'  Tiga fitur teratas ({metode_utama}):')
for _, r in df_pi.head(3).iterrows():
    print(f'    #{int(r["Peringkat"])}: {r["Label"]:<16} = {r["Mean Decrease Recall"]:.4f}')
cocok = ('HbA1c_level' in tiga_teratas[:2]) or ('blood_glucose_level' in tiga_teratas[:2])
print()
print(f'  Ekspektasi klinis: HbA1c dan kadar glukosa berada di peringkat teratas.')
print(f'  Status: {"SESUAI ekspektasi domain medis" if cocok else "TIDAK SESUAI - periksa kembali data dan preprocessing"}')

---

## 8. Checklist Penutup & Verifikasi Deploy

Cell terakhir mencetak daftar file yang dihasilkan, langkah penyalinan ke repo
Laravel, dan checklist verifikasi. Poin paling penting: setelah artefak
disalin, jalankan

```
python model/predict.py 55 0 28.5 6.8 150
```

lalu bandingkan probabilitasnya dengan angka acuan yang dicetak notebook.
Kalau berbeda, berarti `predict.py` masih menyusun array dengan urutan yang
salah atau memuat pickle yang lama.

In [ ]:
# ============================================================
# CELL 14: Checklist Penutup & Verifikasi Deploy
# ============================================================
# Perbarui arsip agar experiments.json ikut masuk
path_zip = shutil.make_archive(f'{OUTPUT_DIR}/artefak_produksi_diapredict', 'zip', PRODUKSI_DIR)

garis('DAFTAR FILE YANG DIHASILKAN NOTEBOOK 06')
print(f'Folder produksi : {PRODUKSI_DIR}')
for nama_file in sorted(os.listdir(PRODUKSI_DIR)):
    ukuran = os.path.getsize(os.path.join(PRODUKSI_DIR, nama_file)) / 1024
    satuan = f'{ukuran/1024:.2f} MB' if ukuran > 1024 else f'{ukuran:.1f} KB'
    print(f'  {nama_file:<26} {satuan}')
print()
print(f'Arsip siap unduh: {path_zip} ({os.path.getsize(path_zip)/(1024**2):.2f} MB)')
print()

print('Tabel (CSV) di ' + OUTPUT_DIR + '/tabel :')
for nama_file in sorted(f for f in os.listdir(f'{OUTPUT_DIR}/tabel') if f.startswith('final_')):
    print(f'  {nama_file}')
print()
print('Gambar (PNG) di ' + OUTPUT_DIR + '/gambar :')
for nama_file in sorted(f for f in os.listdir(f'{OUTPUT_DIR}/gambar') if f.startswith('final_')):
    print(f'  {nama_file}')
print()

# --- Angka acuan untuk verifikasi setelah deploy -----------------------------
# Perhatikan: argumen CLI predict.py berurutan [age, hypertension, bmi, HbA1c, glukosa],
# sedangkan model menerima [age, bmi, hypertension, HbA1c, glukosa].
ARG_CLI = {'age': 55.0, 'hypertension': 0.0, 'bmi': 28.5,
           'HbA1c_level': 6.8, 'blood_glucose_level': 150.0}
vektor_acuan = [ARG_CLI[f] for f in SELECTED_FEATURES]
prob_acuan = prediksi_produksi(vektor_acuan)
label_acuan = 'DIABETES' if prob_acuan >= threshold_final else 'SEHAT'

garis('ANGKA ACUAN VERIFIKASI DEPLOY')
print('Perintah uji:')
print('    python model/predict.py 55 0 28.5 6.8 150')
print('    (urutan argumen CLI: age, hypertension, bmi, HbA1c_level, blood_glucose_level)')
print()
print('Array yang HARUS dibentuk sebelum scaler.transform (urutan training):')
print(f'    {SELECTED_FEATURES}')
print(f'    {vektor_acuan}')
print()
print('Hasil yang harus muncul:')
print(f'    probability : {prob_acuan:.4f}')
print(f'    threshold   : {threshold_final:.4f}')
print(f'    prediction  : {1 if prob_acuan >= threshold_final else 0}  ({label_acuan})')
print()
print(f'Toleransi selisih probabilitas: < 0.0001. Kalau selisihnya besar, berarti')
print(f'predict.py masih memakai urutan lama atau pickle yang belum diganti.')
print()

print('Potongan kode yang benar untuk predict.py:')
print('-' * 70)
print("    FEATURE_ORDER = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']")
print("    nilai = {'age': age, 'hypertension': hypertension, 'bmi': bmi,")
print("             'HbA1c_level': hba1c_level, 'blood_glucose_level': blood_glucose_level}")
print("    input_data = [[nilai[f] for f in FEATURE_ORDER]]   # urutan training, bukan urutan argumen")
print('-' * 70)
print()

garis('CHECKLIST DEPLOY')
checklist = [
    'Backup model/rf_model.pkl dan model/scaler.pkl yang lama (.bak).',
    'Salin rf_model.pkl, scaler.pkl, model_metadata.json ke folder model/.',
    'Salin experiments.json ke public/data/experiments.json.',
    'Ubah predict.py: susun array mengikuti feature_order, bukan urutan argumen CLI.',
    f'Ubah threshold di predict.py dari 0.4965 menjadi {threshold_final:.4f}.',
    'Tambahkan capping winsorization di predict.py memakai preprocessing.winsorization.',
    'Jalankan: python model/predict.py 55 0 28.5 6.8 150',
    f'Pastikan probability yang keluar = {prob_acuan:.4f} (toleransi 0.0001).',
    'Uji tiga pasien contoh dari CELL 10 lewat form website, cocokkan probabilitasnya.',
    'Pastikan versi scikit-learn di server sama dengan versi_library di model_metadata.json.',
    'Cek halaman metodologi di website: angka harus terbaca dari experiments.json.',
    'Pastikan label grafik importance memakai meta.sumber_metode_importance (bukan "SHAP").',
]
for i, item in enumerate(checklist, 1):
    print(f'  [ ] {i:>2}. {item}')
print()
garis('NOTEBOOK 06 SELESAI')
print('Seluruh artefak produksi dan experiments.json sudah dihasilkan.')

---

# RINGKASAN UNTUK SKRIPSI

Notebook 06 menutup rangkaian revisi dengan mengubah hasil eksperimen menjadi
sistem yang benar-benar berjalan. Model final dilatih memakai konfigurasi yang
seluruh komponennya punya dasar: rasio split dari Notebook 01, nilai `k` KNN
dari Notebook 02, kernel dan parameter `C` SVM dari Notebook 03, validasi
statistik dari Notebook 04, serta pemilihan model produksi dari matriks
keputusan Notebook 05. Tidak ada lagi angka yang dipakai hanya karena
"begitu defaultnya".

Model final adalah **Random Forest** dengan hyperparameter hasil `RandomizedSearchCV`
(`n_estimators=200`, `max_depth=10`, `min_samples_split=5`, `min_samples_leaf=4`,
`max_features='log2'`, `criterion='entropy'`, `class_weight='balanced'`), dilatih
di dalam `ImbPipeline` berurutan `StandardScaler -> SMOTE -> RandomForest`.
Urutan tersebut menjamin dua hal sekaligus: SMOTE hanya aktif saat pelatihan
sehingga tidak ada kebocoran data ke set uji, dan `StandardScaler` ter-fit pada
data latih asli sehingga statistiknya mewakili distribusi pasien nyata — persis
kondisi yang dihadapi saat inferensi di website. Threshold keputusan ditetapkan
lewat Youden's J Statistic karena konteks skrining medis menempatkan recall
(kemampuan menangkap kasus diabetes) di atas presisi.

Kontribusi terpenting notebook ini terhadap sisi rekayasa perangkat lunak adalah
**mengunci kontrak antarmuka model**. Sistem lama menderita bug senyap: kode
inferensi menyusun input dengan urutan `[age, hypertension, bmi, ...]` sementara
model dilatih dengan urutan `[age, bmi, hypertension, ...]`, sehingga nilai BMI
dan status hipertensi tertukar pada setiap prediksi. Karena panjang array tetap
lima, program tidak pernah melempar error dan kesalahan itu lolos ke produksi.
CELL 10 mendokumentasikan dampaknya secara kuantitatif, dan `model_metadata.json`
kini memuat field `feature_order` yang eksplisit sebagai satu-satunya rujukan
urutan fitur. Notebook juga mengekspor batas winsorization per fitur, agar
preprocessing di server identik dengan preprocessing saat pelatihan.

Perbaikan berikutnya menyangkut ukuran dan kejujuran pelaporan. Model produksi
lama berukuran sekitar 75 MB karena dilatih dengan `RandomForestClassifier`
default tanpa `max_depth`, sehingga setiap pohon tumbuh sampai daunnya murni.
Model final dengan `max_depth=10` menghasilkan artefak yang jauh lebih ringan
tanpa kehilangan recall, sekaligus konsisten dengan hyperparameter yang
dilaporkan di skripsi. Terakhir, notebook mengoreksi pelabelan XAI: grafik
importance di V2 diberi judul "SHAP" padahal yang dihitung adalah Permutation
Importance pada model KNN. Notebook 06 menghitung ulang importance pada model
final Random Forest — dengan Permutation Importance dan, bila tersedia, TreeSHAP
yang kali ini memang absah — lalu menuliskan metode sebenarnya pada field
`sumber_metode_importance` di `experiments.json` sehingga label yang tampil di
website sesuai dengan perhitungan yang benar-benar dilakukan.

---

# LANGKAH DEPLOY

### 1. Unduh artefak dari Colab

```python
from google.colab import files
files.download(f'{OUTPUT_DIR}/artefak_produksi_diapredict.zip')
```

Isi arsip: `rf_model.pkl`, `scaler.pkl`, `model_metadata.json`, `experiments.json`.

### 2. Backup artefak lama di repo Laravel

```bash
cp model/rf_model.pkl model/rf_model.pkl.bak
cp model/scaler.pkl   model/scaler.pkl.bak
```

### 3. Salin artefak baru

| Dari arsip | Ke repo Laravel |
|---|---|
| `rf_model.pkl` | `model/rf_model.pkl` |
| `scaler.pkl` | `model/scaler.pkl` |
| `model_metadata.json` | `model/model_metadata.json` |
| `experiments.json` | `public/data/experiments.json` |

### 4. Perbaiki `model/predict.py`

Tiga perubahan wajib:

1. **Urutan fitur** — susun array mengikuti `feature_order` dari
   `model_metadata.json`, bukan mengikuti urutan argumen CLI:

   ```python
   FEATURE_ORDER = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
   nilai = {'age': age, 'hypertension': hypertension, 'bmi': bmi,
            'HbA1c_level': hba1c_level, 'blood_glucose_level': blood_glucose_level}
   input_data = [[nilai[f] for f in FEATURE_ORDER]]
   ```

2. **Capping winsorization** — terapkan batas dari
   `model_metadata.json -> preprocessing.winsorization` sebelum `scaler.transform`.

3. **Threshold** — ganti nilai lama `0.4965` dengan threshold hasil notebook ini
   (dicetak di CELL 9 dan CELL 14).

### 5. Verifikasi

```bash
python model/predict.py 55 0 28.5 6.8 150
```

Bandingkan `probability` yang keluar dengan angka acuan pada CELL 14
(toleransi 0.0001). Lanjutkan dengan menguji tiga pasien contoh CELL 10 melalui
form website. Kalau ketiganya cocok, artinya model, scaler, urutan fitur, dan
threshold sudah tersinkronisasi antara notebook dan produksi.

### 6. Periksa halaman metodologi

Pastikan halaman "Metodologi & Pengujian" membaca `public/data/experiments.json`,
dan label grafik feature importance memakai `meta.sumber_metode_importance`
sehingga tidak lagi menyebut "SHAP" untuk perhitungan yang bukan SHAP.

---

*Notebook 06 — Model Final & Export untuk Produksi. Bagian dari Revisi Pengujian V3, DiaPredict.*